# 04 Semantic Deduplication Agent

In [1]:
# ==================================================
# Notebook 04
# Semantic Deduplication Agent
# ==================================================

import numpy as np
import pandas as pd

from typing import TypedDict
from typing import List
from typing import Dict

from sentence_transformers import SentenceTransformer

from sklearn.metrics.pairwise import cosine_similarity

d:\Books\projects-nlp-transformers-learning\.projectnlps\Lib\site-packages\sentence_transformers\cross_encoder\CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


In [2]:
embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

In [3]:
class ResearchState(TypedDict):

    research_goal: str

    queries_executed: List[str]

    raw_findings: List[str]

    deduplicated_findings: List[str]

    conflicts: List[str]

    critic_score: float

    draft_report: str

    final_report: str

    iteration_count: int

    status: str

    errors: List[str]

    metadata: Dict

In [4]:
sample_findings = [
    "AWS pricing increased 8%",
    "AWS cloud costs rose by 8%",
    "Amazon Web Services pricing increased eight percent",
    "Azure pricing remained stable",
    "Azure costs did not change",
    "Google Cloud storage reduced 4%",
]

In [5]:
embeddings = embedding_model.encode(sample_findings)

embeddings.shape

(6, 384)

In [6]:
similarity_matrix = cosine_similarity(embeddings)

similarity_matrix

array([[0.9999999 , 0.80993855, 0.84194434, 0.44675463, 0.4768498 ,
        0.42578295],
       [0.80993855, 1.0000001 , 0.6892378 , 0.42348835, 0.5137893 ,
        0.5597062 ],
       [0.84194434, 0.6892378 , 1.        , 0.41011456, 0.43754113,
        0.4065943 ],
       [0.44675463, 0.42348835, 0.41011456, 1.0000001 , 0.70965666,
        0.3503554 ],
       [0.4768498 , 0.5137893 , 0.43754113, 0.70965666, 1.0000001 ,
        0.3707402 ],
       [0.42578295, 0.5597062 , 0.4065943 , 0.3503554 , 0.3707402 ,
        1.0000001 ]], dtype=float32)

In [7]:
similarity_df = pd.DataFrame(
    similarity_matrix,
    columns=[f"Doc_{i}" for i in range(len(sample_findings))],
    index=[f"Doc_{i}" for i in range(len(sample_findings))],
)

similarity_df

,Doc_0,Doc_1,Doc_2,Doc_3,Doc_4,Doc_5
Doc_0,1.000000,0.809939,0.841944,0.446755,0.476850,0.425783
Doc_1,0.809939,1.000000,0.689238,0.423488,0.513789,0.559706
Doc_2,0.841944,0.689238,1.000000,0.410115,0.437541,0.406594
Doc_3,0.446755,0.423488,0.410115,1.000000,0.709657,0.350355
Doc_4,0.476850,0.513789,0.437541,0.709657,1.000000,0.370740
Doc_5,0.425783,0.559706,0.406594,0.350355,0.370740,1.000000


In [8]:
def semantic_deduplicate(findings, threshold=0.80):

    embeddings = embedding_model.encode(findings)

    similarity_matrix = cosine_similarity(embeddings)

    keep_indices = []

    for i in range(len(findings)):

        is_duplicate = False

        for kept_idx in keep_indices:

            similarity = similarity_matrix[i, kept_idx]

            if similarity >= threshold:

                is_duplicate = True

                break

        if not is_duplicate:

            keep_indices.append(i)

    unique_findings = [findings[i] for i in keep_indices]

    return unique_findings

In [9]:
unique_findings = semantic_deduplicate(sample_findings, threshold=0.80)

unique_findings

['AWS pricing increased 8%',
 'Azure pricing remained stable',
 'Azure costs did not change',
 'Google Cloud storage reduced 4%']

In [10]:
def semantic_deduplicate_with_log(findings, threshold=0.80):

    embeddings = embedding_model.encode(findings)

    similarity_matrix = cosine_similarity(embeddings)

    unique = []

    removed = []

    keep_indices = []

    for i in range(len(findings)):

        duplicate = False

        for kept_idx in keep_indices:

            similarity = similarity_matrix[i, kept_idx]

            if similarity >= threshold:

                duplicate = True

                removed.append(
                    {
                        "removed": findings[i],
                        "matched_with": findings[kept_idx],
                        "similarity": float(similarity),
                    }
                )

                break

        if not duplicate:

            keep_indices.append(i)

            unique.append(findings[i])

    return unique, removed

In [11]:
unique, removed = semantic_deduplicate_with_log(sample_findings)

In [12]:
pd.DataFrame(removed)

,removed,matched_with,similarity
0,AWS cloud costs rose by 8%,AWS pricing increased 8%,0.809939
1,Amazon Web Services pricing increased eight pe...,AWS pricing increased 8%,0.841944


In [13]:
def deduplication_agent(state: ResearchState):

    print("\nDeduplication Agent Running...")

    unique, removed = semantic_deduplicate_with_log(state["raw_findings"])

    state["deduplicated_findings"] = unique

    state["metadata"]["duplicates_removed"] = removed

    state["status"] = "deduplication_completed"

    return state

In [14]:
sample_state = {
    "research_goal": "Compare cloud costs",
    "queries_executed": [],
    "raw_findings": sample_findings,
    "deduplicated_findings": [],
    "conflicts": [],
    "critic_score": 0,
    "draft_report": "",
    "final_report": "",
    "iteration_count": 0,
    "status": "",
    "errors": [],
    "metadata": {},
}

In [15]:
updated_state = deduplication_agent(sample_state)


Deduplication Agent Running...


In [16]:
updated_state["deduplicated_findings"]

['AWS pricing increased 8%',
 'Azure pricing remained stable',
 'Azure costs did not change',
 'Google Cloud storage reduced 4%']

In [17]:
pd.DataFrame(updated_state["metadata"]["duplicates_removed"])

,removed,matched_with,similarity
0,AWS cloud costs rose by 8%,AWS pricing increased 8%,0.809939
1,Amazon Web Services pricing increased eight pe...,AWS pricing increased 8%,0.841944


In [18]:
from langgraph.graph import StateGraph, END

In [19]:
graph = StateGraph(ResearchState)

In [20]:
graph.add_node("deduplicator", deduplication_agent)

In [21]:
graph.set_entry_point("deduplicator")

graph.add_edge("deduplicator", END)

In [22]:
app = graph.compile()

In [23]:
result = app.invoke(sample_state)

result


Deduplication Agent Running...


{'research_goal': 'Compare cloud costs',
 'queries_executed': [],
 'raw_findings': ['AWS pricing increased 8%',
  'AWS cloud costs rose by 8%',
  'Amazon Web Services pricing increased eight percent',
  'Azure pricing remained stable',
  'Azure costs did not change',
  'Google Cloud storage reduced 4%'],
 'deduplicated_findings': ['AWS pricing increased 8%',
  'Azure pricing remained stable',
  'Azure costs did not change',
  'Google Cloud storage reduced 4%'],
 'conflicts': [],
 'critic_score': 0,
 'draft_report': '',
 'final_report': '',
 'iteration_count': 0,
 'status': 'deduplication_completed',
 'errors': [],
 'metadata': {'duplicates_removed': [{'removed': 'AWS cloud costs rose by 8%',
    'matched_with': 'AWS pricing increased 8%',
    'similarity': 0.8099385499954224},
   {'removed': 'Amazon Web Services pricing increased eight percent',
    'matched_with': 'AWS pricing increased 8%',
    'similarity': 0.8419443368911743}]}}

{'research_goal': 'Compare cloud costs against competitors',
 'queries_executed': ['cloud pricing comparison'],
 'raw_findings': ['AWS pricing increased 8%', 'Azure pricing remained stable'],
 'deduplicated_findings': [],
 'conflicts': [],
 'critic_score': 0.0,
 'draft_report': '',
 'final_report': '',
 'iteration_count': 0,
 'status': 'research_completed'}

['AWS pricing increased 8%', 'Azure pricing remained stable']

0.4

AWS pricing increased 8%
Azure pricing remained stable


1

1

{'research_goal': 'Compare cloud costs against competitors',
 'queries_executed': ['cloud pricing comparison'],
 'raw_findings': ['AWS pricing increased 8%', 'Azure pricing remained stable'],
 'deduplicated_findings': ['AWS pricing increased 8%',
  'Azure pricing remained stable'],
 'conflicts': [],
 'critic_score': 0.4,
 'draft_report': 'AWS pricing increased 8%\nAzure pricing remained stable',
 'final_report': '',
 'iteration_count': 1,
 'status': 'draft_created'}

{'research_goal': 'Compare cloud costs against competitors',
 'queries_executed': ['cloud pricing comparison'],
 'raw_findings': ['AWS pricing increased 8%', 'Azure pricing remained stable'],
 'deduplicated_findings': ['AWS pricing increased 8%',
  'Azure pricing remained stable'],
 'conflicts': [],
 'critic_score': 0.4,
 'draft_report': 'AWS pricing increased 8%\nAzure pricing remained stable',
 'final_report': '',
 'iteration_count': 1,
 'status': 'draft_created',
 'errors': ['Search API timeout']}